# Dart candidates from rotational differences

This notebook tests whether a board-plane top-down transform followed by a 36° rotation can suppress the dartboard's repeating geometry and expose dart-shaped residuals.

Residuals are candidate evidence, not a tip detector. The board homography rectifies the board plane, not the protruding dart; the rotation also creates a rotated copy (ghost) of the dart. Lighting, print, interpolation, and calibration errors create additional differences.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from dartcv.detect import detect_board
from dartcv.diagnostics import detection_overlays
from dartcv.edge_detection import canny_edges
from dartcv.geometry import RADII
from dartcv.overlay import draw_overlay, model_to_topdown

cwd = Path.cwd().resolve()
repo_root = next(
    (path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the dartcv repository root")

IMAGE_PATH = repo_root / "data" / "images" / "single_dart" / "PXL_20261018_152634348.jpg"
UPRIGHT_DEG = 0.0
ROTATION_DEG = 36.0
SEARCH_MARGIN_MM = 100.0
CANNY_LOW = 50
CANNY_HIGH = 150
BLUR_KERNEL = (5, 5)
BLUR_SIGMA = 0.0
MIN_COMPONENT_AREA = 30
MAX_CANDIDATES = 40
TIP_POINT_TOPDOWN = None


def show_grid(images, titles, cmaps=None, max_cols=2, panel_size=(8, 6)):
    if cmaps is None:
        cmaps = [None] * len(images)
    ncols = min(max_cols, len(images))
    nrows = (len(images) + ncols - 1) // ncols
    figure, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(panel_size[0] * ncols, panel_size[1] * nrows),
        squeeze=False,
    )
    for axis, image_view, title, cmap in zip(axes.flat, images, titles, cmaps):
        axis.imshow(image_view, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    for axis in axes.flat[len(images):]:
        axis.set_visible(False)
    figure.tight_layout()
    plt.show()
    return figure, axes


def bgr_to_rgb(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

## Load one image

The selected photograph stays in the original BGR coordinate frame. All later coordinates and transforms are derived from this image.

In [ ]:
image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image is None:
    raise FileNotFoundError(IMAGE_PATH)

print(f"Image: {IMAGE_PATH.relative_to(repo_root)}")
print(f"Source dimensions: {image.shape[1]} x {image.shape[0]}")
show_grid([bgr_to_rgb(image)], ["Source image"], max_cols=1, panel_size=(10, 8))

## Detect and calibrate the board

The detector estimates the board-to-image homography `H`. The overlay and diagnostic stages are checks on that estimate, not annotations of ground-truth geometry.

In [ ]:
detection = detect_board(image, upright_deg=UPRIGHT_DEG, diagnostics=True)
diagnostic_views = detection_overlays(image, detection)
ring_contours = {name: ring.contour for name, ring in detection.rings.items()}
source_overlay = draw_overlay(
    image,
    detection.H,
    thickness=3,
    ring_contours=ring_contours,
)

print(f"Estimated 20-sector orientation: {detection.rotation_deg:.1f} degrees")
print(f"Bull independently detected: {detection.bull is not None}")
print(f"Accepted ECC correlation: {detection.ecc_score}")
show_grid(
    [
        bgr_to_rgb(diagnostic_views["01_colour_mask.png"]),
        bgr_to_rgb(diagnostic_views["02_outer_ellipse.png"]),
        bgr_to_rgb(diagnostic_views["05_calibration.png"]),
        bgr_to_rgb(source_overlay),
    ],
    ["Colour mask", "Outer ellipse fit", "Projected calibration wires", "Final source overlay"],
    panel_size=(8, 6),
)

## Rectify and build an expanded search mask

The scoring circle ends at the nominal outer double ring. Dart flights can extend beyond it, so this notebook creates a larger custom top-down canvas and a separate circular search mask. `SEARCH_MARGIN_MM` controls how far the search region extends beyond the outer-double radius; the numbers and their surrounding ring remain inside it.

In [ ]:
if SEARCH_MARGIN_MM < 0:
    raise ValueError("SEARCH_MARGIN_MM must be nonnegative")

pixels_per_mm = model_to_topdown()[0, 0]
scoring_radius_mm = RADII["double_outer"]
search_radius_mm = scoring_radius_mm + SEARCH_MARGIN_MM
canvas_size = int(np.ceil(2 * search_radius_mm * pixels_per_mm))
height = width = canvas_size
center = (width / 2.0, height / 2.0)
model_to_search = np.array(
    [
        [pixels_per_mm, 0, center[0]],
        [0, pixels_per_mm, center[1]],
        [0, 0, 1],
    ],
    dtype=np.float64,
)
source_to_search = model_to_search @ np.linalg.inv(detection.H)
topdown = cv2.warpPerspective(image, source_to_search, (width, height))

yy, xx = np.ogrid[:height, :width]
radius_squared = (xx - center[0]) ** 2 + (yy - center[1]) ** 2
scoring_radius_px = scoring_radius_mm * pixels_per_mm
search_radius_px = search_radius_mm * pixels_per_mm
scoring_mask = (radius_squared <= scoring_radius_px**2).astype(np.uint8) * 255
search_mask = (radius_squared <= search_radius_px**2).astype(np.uint8) * 255
masked_topdown = cv2.bitwise_and(topdown, topdown, mask=search_mask)

mask_overlay = topdown.copy()
cv2.circle(
    mask_overlay,
    tuple(np.round(center).astype(int)),
    int(round(scoring_radius_px)),
    (0, 255, 0),
    3,
)
cv2.circle(
    mask_overlay,
    tuple(np.round(center).astype(int)),
    int(round(search_radius_px)),
    (0, 165, 255),
    3,
)
print(f"Expanded top-down canvas: {width} x {height} px")
print(f"Scoring radius: {scoring_radius_mm:.1f} mm")
print(f"Search radius: {search_radius_mm:.1f} mm")
show_grid(
    [bgr_to_rgb(topdown), search_mask, bgr_to_rgb(mask_overlay)],
    ["Expanded top-down image", "Expanded search mask", "Scoring circle (green) and search boundary (orange)"],
    cmaps=[None, "gray", None],
    panel_size=(8, 7),
)

## Rotate a reference by 36°

Rotation is around the bull at the canvas center. Only pixels valid in both the original and rotated playable masks will be compared.

In [ ]:
rotation_matrix = cv2.getRotationMatrix2D(center, ROTATION_DEG, 1.0)
rotated_topdown = cv2.warpAffine(
    topdown,
    rotation_matrix,
    (width, height),
    flags=cv2.INTER_LINEAR,
    borderMode=cv2.BORDER_CONSTANT,
)
rotated_mask = cv2.warpAffine(
    search_mask,
    rotation_matrix,
    (width, height),
    flags=cv2.INTER_NEAREST,
    borderMode=cv2.BORDER_CONSTANT,
)
comparison_mask = cv2.bitwise_and(search_mask, rotated_mask)
rotated_masked = cv2.bitwise_and(
    rotated_topdown, rotated_topdown, mask=comparison_mask
)
original_masked = cv2.bitwise_and(topdown, topdown, mask=comparison_mask)

show_grid(
    [bgr_to_rgb(original_masked), bgr_to_rgb(rotated_masked)],
    ["Expanded board + dart search region", f"Reference rotated by {ROTATION_DEG:g}°"],
    panel_size=(8, 7),
)

## Compare grayscale and edge residuals

Grayscale differences are sensitive to illumination. Edge XOR emphasizes structural changes, but fine print and registration error can also survive. The rotated dart may appear as a second residual, so inspect both before treating anything as a candidate.

In [ ]:
gray_original = cv2.cvtColor(original_masked, cv2.COLOR_BGR2GRAY)
gray_rotated = cv2.cvtColor(rotated_masked, cv2.COLOR_BGR2GRAY)
gray_difference = cv2.absdiff(gray_original, gray_rotated)

comparison_mask_inner = cv2.erode(
    comparison_mask,
    np.ones((9, 9), np.uint8),
    iterations=1,
)
gray_difference[comparison_mask_inner == 0] = 0
_, gray_difference_binary = cv2.threshold(
    gray_difference, 25, 255, cv2.THRESH_BINARY
)

edges_original = canny_edges(
    original_masked,
    low_threshold=CANNY_LOW,
    high_threshold=CANNY_HIGH,
    blur_kernel=BLUR_KERNEL,
    blur_sigma=BLUR_SIGMA,
)
edges_rotated = cv2.warpAffine(
    edges_original,
    rotation_matrix,
    (width, height),
    flags=cv2.INTER_NEAREST,
    borderMode=cv2.BORDER_CONSTANT,
)
edge_difference = cv2.bitwise_xor(edges_original, edges_rotated)
edge_difference[comparison_mask_inner == 0] = 0

show_grid(
    [
        gray_difference,
        gray_difference_binary,
        edges_original,
        edges_rotated,
        edge_difference,
    ],
    [
        "Grayscale absolute difference",
        "Thresholded grayscale difference",
        "Original Canny edges",
        "Rotated Canny edges",
        "Canny XOR residual",
    ],
    cmaps=["gray"] * 5,
    panel_size=(8, 6),
)

## Inspect connected residual candidates

Connected components provide regions to inspect, not tip predictions. The minimum area and maximum bounding-box aspect ratio are adjustable; board printing and registration errors may still produce candidates.

In [ ]:
component_count, component_labels, component_stats, component_centroids = (
    cv2.connectedComponentsWithStats(
        (edge_difference > 0).astype(np.uint8), connectivity=8
    )
)
components = []
for label in range(1, component_count):
    x, y, box_width, box_height, area = component_stats[label]
    long_side = max(box_width, box_height)
    short_side = max(1, min(box_width, box_height))
    aspect_ratio = long_side / short_side
    if area >= MIN_COMPONENT_AREA and long_side >= 12 and aspect_ratio <= 20:
        components.append(
            (int(area), int(x), int(y), int(box_width), int(box_height), int(label))
        )

components.sort(reverse=True)
components = components[:MAX_CANDIDATES]
candidate_view = cv2.cvtColor(edge_difference, cv2.COLOR_GRAY2BGR)
for index, (area, x, y, box_width, box_height, label) in enumerate(components, start=1):
    cv2.rectangle(
        candidate_view,
        (x, y),
        (x + box_width, y + box_height),
        (0, 165, 255),
        2,
    )
    cv2.putText(
        candidate_view,
        str(index),
        (x, max(14, y - 4)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 165, 255),
        1,
        cv2.LINE_AA,
    )

print(f"Connected residual components: {component_count - 1}")
print(f"Components passing simple size filters: {len(components)}")
print("Candidate rows: rank, area, x, y, width, height, label")
for index, component in enumerate(components, start=1):
    print(index, *component)
show_grid(
    [edge_difference, bgr_to_rgb(candidate_view)],
    ["Canny XOR residual", "Size-filtered candidate boxes"],
    cmaps=["gray", None],
    panel_size=(9, 7),
)

## Optional provisional point

After inspecting the residual and shaft direction, set `TIP_POINT_TOPDOWN = (x, y)` in the settings cell to annotate a candidate entry point in top-down pixels. The inverse board transform maps it back to source-image pixels for comparison only; because the dart protrudes from the board plane, this mapping is approximate.

In [ ]:
if TIP_POINT_TOPDOWN is None:
    print(
        "No point is inferred automatically. Inspect the shaft/entry region, "
        "then optionally set TIP_POINT_TOPDOWN in the settings cell."
    )
else:
    tip_x, tip_y = map(float, TIP_POINT_TOPDOWN)
    if not (0 <= tip_x < width and 0 <= tip_y < height):
        raise ValueError("TIP_POINT_TOPDOWN must lie inside the expanded search canvas")
    if search_mask[int(round(tip_y)), int(round(tip_x))] == 0:
        raise ValueError("TIP_POINT_TOPDOWN must lie inside the expanded search mask")

    point = np.array([[[tip_x, tip_y]]], dtype=np.float32)
    point_source = cv2.perspectiveTransform(
        point, np.linalg.inv(source_to_search)
    )[0, 0]

    residual_with_point = cv2.cvtColor(edge_difference, cv2.COLOR_GRAY2BGR)
    cv2.drawMarker(
        residual_with_point,
        (round(tip_x), round(tip_y)),
        (0, 0, 255),
        cv2.MARKER_CROSS,
        24,
        2,
    )
    source_with_point = source_overlay.copy()
    cv2.drawMarker(
        source_with_point,
        tuple(np.round(point_source).astype(int)),
        (0, 0, 255),
        cv2.MARKER_CROSS,
        30,
        3,
    )
    print(
        "Provisional mapped source point (x, y): "
        f"({point_source[0]:.1f}, {point_source[1]:.1f})"
    )
    show_grid(
        [bgr_to_rgb(residual_with_point), bgr_to_rgb(source_with_point)],
        ["Provisional point on residual", "Approximate source-image projection"],
        panel_size=(9, 7),
    )

## Interpretation limits

A 36° comparison can reduce the repeating sector/wire pattern, but does not make the full board or lighting exactly symmetric. Printed marks, occlusion, interpolation, and calibration error remain in the residual. The rotated dart produces a ghost, and the true impact point may be hidden. This notebook surfaces candidate regions; it does not automatically identify a ground-truth dart tip.

## Compact batch: final outputs only

Set `RUN_BATCH = True` and edit `BATCH_PATHS` to select any subset (remove `[:3]`
for the full directory). Run this cell again without repeating the intermediate
visualizations. Processing is sequential; pages retain small output thumbnails,
not full-resolution diagnostic histories. Each tile names the input, final
status and processing time. Failures are visible, not silently skipped.

To investigate a bad tile, copy its filename into this notebook's single-image
selection and rerun the detailed sections. Batch mode uses automatic calibration;
never reuse one image's manual points/tips on another image.

Runs the rotation/Canny-XOR/size-filter experiment with explicit batch settings. Candidate boxes are residual components, not verified darts.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from dartcv.notebook_batch import BatchOutput, image_batch_figures, read_batch_image

BATCH_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())

RUN_BATCH = False
BATCH_PATHS = sorted((BATCH_ROOT / "data/images/single_dart").glob("*.jpg"))[:3]

from dartcv.detect import detect_board
from dartcv.edge_detection import canny_edges
from dartcv.geometry import RADII
from dartcv.overlay import model_to_topdown

BATCH_ROTATION_DEG = 36.0
BATCH_SEARCH_MARGIN_MM = 100.0
BATCH_CANNY_LOW, BATCH_CANNY_HIGH = 50, 150
BATCH_MIN_AREA, BATCH_MAX_CANDIDATES = 30, 40

def batch_output(path):
    source = read_batch_image(path)
    detection = detect_board(source)
    scale = model_to_topdown()[0, 0]
    radius = RADII["double_outer"] + BATCH_SEARCH_MARGIN_MM
    size = int(np.ceil(2 * radius * scale))
    center = (size / 2, size / 2)
    transform = np.array([[scale, 0, center[0]], [0, scale, center[1]], [0, 0, 1.]])
    topdown = cv2.warpPerspective(source, transform @ np.linalg.inv(detection.H), (size, size))
    yy, xx = np.ogrid[:size, :size]
    mask = (((xx-center[0])**2 + (yy-center[1])**2) <= (radius*scale)**2).astype(np.uint8)*255
    rotation = cv2.getRotationMatrix2D(center, BATCH_ROTATION_DEG, 1)
    rotated_mask = cv2.warpAffine(mask, rotation, (size, size), flags=cv2.INTER_NEAREST)
    common = cv2.erode(cv2.bitwise_and(mask, rotated_mask), np.ones((9, 9), np.uint8))
    masked = cv2.bitwise_and(topdown, topdown, mask=cv2.bitwise_and(mask, rotated_mask))
    edges = canny_edges(masked, low_threshold=BATCH_CANNY_LOW,
                        high_threshold=BATCH_CANNY_HIGH, blur_kernel=(5, 5), blur_sigma=0)
    rotated_edges = cv2.warpAffine(edges, rotation, (size, size), flags=cv2.INTER_NEAREST)
    residual = cv2.bitwise_xor(edges, rotated_edges)
    residual[common == 0] = 0
    count, labels, stats, centers = cv2.connectedComponentsWithStats((residual > 0).astype(np.uint8), connectivity=8)
    candidates = []
    for label in range(1, count):
        x, y, width, height, area = stats[label]
        if area >= BATCH_MIN_AREA and max(width, height) >= 12 and max(width, height)/max(1, min(width, height)) <= 20:
            candidates.append((int(area), int(x), int(y), int(width), int(height)))
    output = cv2.cvtColor(residual, cv2.COLOR_GRAY2BGR)
    for area, x, y, width, height in sorted(candidates, reverse=True)[:BATCH_MAX_CANDIDATES]:
        cv2.rectangle(output, (x, y), (x+width, y+height), (0, 165, 255), 2)
    return BatchOutput(output, f"Rotation-XOR candidates: {min(len(candidates), BATCH_MAX_CANDIDATES)}; not dart tips")

if RUN_BATCH:
    for batch_figure in image_batch_figures(BATCH_PATHS, batch_output, page_size=6):
        display(batch_figure)
        plt.close(batch_figure)